<a href="https://colab.research.google.com/github/prernaprachi96/ReviewShield/blob/main/ReviewShield.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [22]:
# Cell 1: Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import string
import warnings
warnings.filterwarnings('ignore')

# Text preprocessing
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

# Machine learning
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_recall_fscore_support
from sklearn.metrics import roc_curve, auc, roc_auc_score

# Models
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier
from xgboost import XGBClassifier
from sklearn.neighbors import KNeighborsClassifier

# Deep learning
from sklearn.neural_network import MLPClassifier

# Download required NLTK data
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('punkt_tab', quiet=True)

print("Libraries imported successfully!")

Libraries imported successfully!


In [23]:
import pandas as pd

df = pd.read_excel("fake-reviews-dataset.xlsx")

In [ ]:
import pandas as pd

df = pd.read_excel("fake-reviews-dataset.xlsx")

In [24]:
df = df[['text_', 'label']]  # keep only required columns
df.dropna(inplace=True)
df.reset_index(drop=True, inplace=True)

print(df.head())

                                               text_ label
0  Love this!  Well made, sturdy, and very comfor...    CG
1  love it, a great upgrade from the original.  I...    CG
2  This pillow saved my back. I love the look and...    CG
3  Missing information on how to use it, but it i...    CG
4  Very nice set. Good quality. We have had the s...    CG


In [25]:
# Cell 4: Data Preprocessing - Text Cleaning
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
import re

# Download required NLTK data
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('punkt', quiet=True)

# Initialize lemmatizer and stopwords
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def clean_text(text):
    """Clean and preprocess text"""
    if pd.isna(text):
        return ""

    # Convert to lowercase
    text = text.lower()

    # Remove URLs
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)

    # Remove mentions and hashtags
    text = re.sub(r'@\w+|#\w+', '', text)

    # Remove punctuation and numbers
    text = re.sub(r'[^\w\s]', '', text)
    text = re.sub(r'\d+', '', text)

    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()

    # Tokenize
    tokens = word_tokenize(text)

    # Remove stopwords and lemmatize
    tokens = [lemmatizer.lemmatize(token) for token in tokens if token not in stop_words and len(token) > 2]

    return ' '.join(tokens)

# Apply text cleaning
df['cleaned_text'] = df['text_'].apply(clean_text)

print("Text cleaning completed!")
print(f"\nOriginal text: {df['text_'].iloc[0]}")
print(f"Cleaned text: {df['cleaned_text'].iloc[0]}")

Text cleaning completed!

Original text: Love this!  Well made, sturdy, and very comfortable.  I love it!Very pretty
Cleaned text: love well made sturdy comfortable love itvery pretty


In [27]:
#Train-Test Split
from sklearn.model_selection import train_test_split

X = df['text_']
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Train size:", len(X_train))
print("Test size:", len(X_test))

Train size: 32345
Test size: 8087


In [31]:
# TF-IDF Vectorization
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(max_features=5000)

X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

print("Vectorization completed")

Vectorization completed


In [32]:
# Logistic Regression Model
from sklearn.linear_model import LogisticRegression

lr_model = LogisticRegression(max_iter=1000)
lr_model.fit(X_train_vec, y_train)

print("Logistic Regression training completed")

Logistic Regression training completed


In [33]:
# Evaluation - Logistic Regression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

y_pred_lr = lr_model.predict(X_test_vec)

print("Logistic Regression Accuracy:", accuracy_score(y_test, y_pred_lr))
print("\nClassification Report:\n", classification_report(y_test, y_pred_lr))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred_lr))

Logistic Regression Accuracy: 0.8984790404352665

Classification Report:
               precision    recall  f1-score   support

          CG       0.90      0.89      0.90      4016
          OR       0.89      0.90      0.90      4071

    accuracy                           0.90      8087
   macro avg       0.90      0.90      0.90      8087
weighted avg       0.90      0.90      0.90      8087


Confusion Matrix:
 [[3584  432]
 [ 389 3682]]


In [34]:
# Naive Bayes Model
from sklearn.naive_bayes import MultinomialNB

nb_model = MultinomialNB()
nb_model.fit(X_train_vec, y_train)

print("Naive Bayes training completed")

Naive Bayes training completed


In [35]:
# Evaluation - Naive Bayes
y_pred_nb = nb_model.predict(X_test_vec)

print("Naive Bayes Accuracy:", accuracy_score(y_test, y_pred_nb))
print("\nClassification Report:\n", classification_report(y_test, y_pred_nb))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred_nb))

Naive Bayes Accuracy: 0.8695437121305799

Classification Report:
               precision    recall  f1-score   support

          CG       0.85      0.90      0.87      4016
          OR       0.89      0.84      0.87      4071

    accuracy                           0.87      8087
   macro avg       0.87      0.87      0.87      8087
weighted avg       0.87      0.87      0.87      8087


Confusion Matrix:
 [[3604  412]
 [ 643 3428]]


In [36]:
# SVM Model
from sklearn.svm import LinearSVC

svm_model = LinearSVC()
svm_model.fit(X_train_vec, y_train)

print("SVM training completed")

SVM training completed


In [37]:
# Evaluation - SVM
y_pred_svm = svm_model.predict(X_test_vec)

print("SVM Accuracy:", accuracy_score(y_test, y_pred_svm))
print("\nClassification Report:\n", classification_report(y_test, y_pred_svm))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred_svm))

SVM Accuracy: 0.9046618028935328

Classification Report:
               precision    recall  f1-score   support

          CG       0.90      0.91      0.90      4016
          OR       0.91      0.90      0.90      4071

    accuracy                           0.90      8087
   macro avg       0.90      0.90      0.90      8087
weighted avg       0.90      0.90      0.90      8087


Confusion Matrix:
 [[3650  366]
 [ 405 3666]]


In [42]:
# Random Forest Model
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train_vec, y_train)

print("Random Forest training completed")

Random Forest training completed


In [43]:
# Evaluation - Random Forest
y_pred_rf = rf_model.predict(X_test_vec)

print("Random Forest Accuracy:", accuracy_score(y_test, y_pred_rf))
print("\nClassification Report:\n", classification_report(y_test, y_pred_rf))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred_rf))

Random Forest Accuracy: 0.8838877210337579

Classification Report:
               precision    recall  f1-score   support

          CG       0.87      0.90      0.88      4016
          OR       0.90      0.87      0.88      4071

    accuracy                           0.88      8087
   macro avg       0.88      0.88      0.88      8087
weighted avg       0.88      0.88      0.88      8087


Confusion Matrix:
 [[3603  413]
 [ 526 3545]]


In [44]:
# Model Comparison
print("\n--- Model Comparison ---")
print("Logistic Regression:", accuracy_score(y_test, y_pred_lr))
print("Naive Bayes:", accuracy_score(y_test, y_pred_nb))
print("SVM:", accuracy_score(y_test, y_pred_svm))
print("Random Forest:", accuracy_score(y_test, y_pred_rf))


--- Model Comparison ---
Logistic Regression: 0.8984790404352665
Naive Bayes: 0.8695437121305799
SVM: 0.9046618028935328
Random Forest: 0.8838877210337579


In [45]:
import pickle

# Save vectorizer
pickle.dump(vectorizer, open("tfidf_vectorizer.pkl", "wb"))

# Save model (SVM)
pickle.dump(svm_model, open("fake_review_model.pkl", "wb"))

print("Model and vectorizer saved successfully")

Model and vectorizer saved successfully


In [46]:
# Label meaning
label_map = {
    "CG": "Fake Review",
    "OR": "Genuine Review"
}

# Test reviews
sample_reviews = [
    "This product is amazing, I loved it!",
    "Worst product ever, completely useless and fake"
]

# Transform and predict
sample_vec = vectorizer.transform(sample_reviews)
predictions = svm_model.predict(sample_vec)

# Print results
for i in range(len(sample_reviews)):
    print("\nReview:", sample_reviews[i])
    print("Prediction:", predictions[i])
    print("Meaning:", label_map[predictions[i]])


Review: This product is amazing, I loved it!
Prediction: CG
Meaning: Fake Review

Review: Worst product ever, completely useless and fake
Prediction: OR
Meaning: Genuine Review


In [47]:
# Load saved model and vectorizer
loaded_vectorizer = pickle.load(open("tfidf_vectorizer.pkl", "rb"))
loaded_model = pickle.load(open("fake_review_model.pkl", "rb"))

# Test again
test_review = ["This is a fake review, do not trust it"]

test_vec = loaded_vectorizer.transform(test_review)
pred = loaded_model.predict(test_vec)

print("\nLoaded Model Prediction:", pred[0])
print("Meaning:", label_map[pred[0]])


Loaded Model Prediction: CG
Meaning: Fake Review


In [50]:
!pip install gradio

In [53]:
import gradio as gr
import pickle

# Load model and vectorizer
vectorizer = pickle.load(open("tfidf_vectorizer.pkl", "rb"))
model = pickle.load(open("fake_review_model.pkl", "rb"))

# Label mapping
label_map = {
    "CG": "✅ Genuine Review",
    "OR": "❌ Fake Review"
}

# Prediction function
def predict_review(review):
    if review.strip() == "":
        return "⚠️ Please enter a review"

    vec = vectorizer.transform([review])
    pred = model.predict(vec)[0]

    return label_map[pred]

# UI Design
interface = gr.Interface(
    fn=predict_review,
    inputs=gr.Textbox(lines=5, placeholder="Enter product review here..."),
    outputs="text",
    title="🛒 Fake Review Detection System",
    description="Enter a product review and check whether it is Fake or Genuine",
)

# Launch app
interface.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://f5ef47ef1f0d9f4356.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
